In [2]:
import torch
from transformers import AutoModel, AutoTokenizer

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_name = "bert-base-uncased"  # 🔄 switched from xlm-roberta-base

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

model = model.to(device)

Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.weight', 'cls.predictions.bias', 'cls.predictions.transform.dense.weight', 'cls.predictions.transform.LayerNorm.weight', 'cls.predictions.decoder.weight', 'cls.seq_relationship.bias', 'cls.predictions.transform.dense.bias']
- This IS expected if you are initializing BertModel from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertModel from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [20]:
# Prepare input for XLM-Roberta
def prepare_xlmr_inputs(text):

    encoding = tokenizer(
        text,
        return_tensors="pt",
        padding='max_length',   # Automatically pad to max_length
        truncation=True,        # Truncate if text too long
        max_length=512          # Max token length
    )
    
    input_ids = encoding['input_ids']
    attention_mask = encoding['attention_mask']
    
    return input_ids.to(device), attention_mask.to(device)


input_ids, attention_mask = prepare_xlmr_inputs("이 사이트는 뉴스 기사를 제공합니다.")

print("input_ids.shape:", input_ids.shape)
print("attention_mask.shape:", attention_mask.shape)
print("First 10 input_ids:", input_ids[0][:10])


input_ids.shape: torch.Size([1, 512])
attention_mask.shape: torch.Size([1, 512])
First 10 input_ids: tensor([     0,   1504,  36667,    769,  46454,  61043,    688, 150811,      5,
             2], device='cuda:0')


In [21]:
@torch.no_grad()
def get_xlmr_embedding(text):
    model.eval()
    
    input_ids, attention_mask = prepare_xlmr_inputs(text)
    input_ids = input_ids.to(device)
    attention_mask = attention_mask.to(device)

    
    outputs = model(input_ids=input_ids, attention_mask=attention_mask)
    
   
    cls_embedding = outputs.last_hidden_state[:, 0, :]  

    return cls_embedding.squeeze(0) 


test_text = "이 사이트는 최신 뉴스와 사건을 다룹니다."
emb = get_xlmr_embedding(test_text)


print("Embedding shape:", emb.shape)
print("First 5 values:", emb[:5])


Embedding shape: torch.Size([768])
First 5 values: tensor([ 0.1624,  0.1323,  0.0753, -0.0129,  0.0297], device='cuda:0')


In [22]:
import torch.nn as nn

class Classifier(nn.Module):
    def __init__(self, num_classes):
        super(Classifier, self).__init__()
        self.fc1 = nn.Linear(768, 512)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(512, num_classes)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x


In [23]:
import pandas as pd


df = pd.read_csv("../make-dataset/cleaned_small_websites_texts.csv", encoding='utf-8')


def split_text_to_chunks(text, chunk_word_limit=50):
    words = str(text).split()
    return [' '.join(words[i:i+chunk_word_limit]) for i in range(0, len(words), chunk_word_limit)]

texts = []
labels = []

website_chunks = []

for i, row in df.iterrows():
    full_text = row["cleaned_website_text"]
    category = row["Category"]

   
    if category in ["도박"]:
        continue

    chunks = split_text_to_chunks(full_text)

    texts.extend(chunks)
    labels.extend([category] * len(chunks))

    website_chunks.append({
        "index": i,
        "label": category,
        "chunk_count": len(chunks),
        "chunks": chunks
    })


print(" 총 웹사이트 개수:", len(website_chunks))
print()

for site in website_chunks[:3]:
    print(f" Index: {site['index']} | Label: {site['label']} | Chunks: {site['chunk_count']}")
    print(f" Example chunk: {site['chunks'][0][:100]}...\n")


 총 웹사이트 개수: 192

 Index: 0 | Label: 보험 | Chunks: 40
 Example chunk: KYOBO 교보생명 본문 바로가기 교보생명 인증센터 소비자포털 회사소개 공시실 ENGLISH 개인 기업 공시실 회사소개 Corporate Information MY교보 나의 보험 ...

 Index: 1 | Label: 보험 | Chunks: 11
 Example chunk: 삼성화재 본문으로 건너뛰기 삼성화재 메인페이지 당신에게 좋은보험 삼성화재 서비스링크 기업보험 삼성화재 다이렉트 회사소개 IR 공시실 소비자포털 RC 보험설계사 회원 가입 로그인 회...

 Index: 2 | Label: 보험 | Chunks: 48
 Example chunk: 현대해상 본문내용 바로가기 대표 기업 적하 퇴직연금 다이렉트 회사소개 소비자포털 공시실 디지털파트너센터 펀드시스템 큰글씨 화면 확대 화면 축소 현대해상 개인 법인 주요메뉴 인터넷창...



In [24]:
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


all_labels = sorted(set(site["label"] for site in website_chunks))
label_to_idx = {label: idx for idx, label in enumerate(all_labels)}
idx_to_label = {v: k for k, v in label_to_idx.items()}


site_embeddings = []
site_labels = []

for site in website_chunks:
    chunk_embeddings = []
    for chunk in site["chunks"]:
        emb = get_xlmr_embedding(chunk)  
        emb = emb.to('cpu')
        chunk_embeddings.append(emb)

    avg_embedding = torch.stack(chunk_embeddings).mean(dim=0)
    site_embeddings.append(avg_embedding)
    site_labels.append(label_to_idx[site["label"]])


embeddings = torch.stack(site_embeddings).to(device)
labels_tensor = torch.tensor(site_labels).to(device)

print("Embedding shape:", embeddings.shape)
print("Label tensor shape:", labels_tensor.shape)
print("Label for site 0:", labels_tensor[0].item(), "(", idx_to_label[labels_tensor[0].item()], ")")


Embedding shape: torch.Size([192, 768])
Label tensor shape: torch.Size([192])
Label for site 0: 2 ( 보험 )


In [25]:
import pandas as pd


label_names = [idx_to_label[label] for label in labels_tensor.cpu().numpy()]
df_labels = pd.DataFrame({'Category': label_names})


class_counts = df_labels["Category"].value_counts()

print(f" Total classes: {len(class_counts)}")
print(" Class list with counts:")
for label, count in class_counts.items():
    print(f"  {label:<15}: {count} samples")


 Total classes: 5
 Class list with counts:
  보험             : 46 samples
  게임             : 46 samples
  금융             : 44 samples
  주식             : 39 samples
  코인             : 17 samples


In [26]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split


class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, smoothing=0.1):
        super(LabelSmoothingCrossEntropy, self).__init__()
        self.smoothing = smoothing

    def forward(self, pred, target):
        num_classes = pred.size(1)
        log_preds = nn.functional.log_softmax(pred, dim=1)
        with torch.no_grad():
            true_dist = torch.zeros_like(pred)
            true_dist.fill_(self.smoothing / (num_classes - 1))
            true_dist.scatter_(1, target.data.unsqueeze(1), 1.0 - self.smoothing)
        return torch.mean(torch.sum(-true_dist * log_preds, dim=1))


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model_mlp = Classifier(num_classes=len(label_to_idx)).to(device)  
loss_fn = LabelSmoothingCrossEntropy(smoothing=0.1)
optimizer = optim.Adam(model_mlp.parameters(), lr=1e-3)


full_dataset = TensorDataset(embeddings.to(device), labels_tensor.to(device))


all_indices = list(range(len(full_dataset)))
train_indices, val_indices = train_test_split(
    all_indices,
    test_size=0.2,
    random_state=42,
    stratify=labels_tensor.cpu()
)


train_dataset = torch.utils.data.Subset(full_dataset, train_indices)
val_dataset = torch.utils.data.Subset(full_dataset, val_indices)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8)


In [27]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np


all_labels = labels_tensor.cpu().numpy()

class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.arange(len(label_to_idx)),  # [0, 1, 2]
    y=all_labels
)

class_weights_tensor = torch.tensor(class_weights, dtype=torch.float).to(device)


In [29]:
# 2. Update loss function with class weights
loss_fn = nn.CrossEntropyLoss(weight=class_weights_tensor)

# Training loop with smarter early stopping
best_val_loss = float('inf')
best_val_acc = 0.0
patience = 5
patience_counter = 0
reached_acc_threshold = False

print("\n🚀 Starting training...\n")
print(f"{'Epoch':<6} {'Train Loss':<12} {'Train Acc':<12} {'Val Loss':<12} {'Val Acc':<12} Status")

for epoch in range(1, 70):
    model_mlp.train()
    total_loss = 0
    correct_train = 0
    total_train = 0

    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()
        preds = model_mlp(x_batch)
        loss = loss_fn(preds, y_batch)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

        predicted = torch.argmax(preds, dim=1)
        correct_train += (predicted == y_batch).sum().item()
        total_train += y_batch.size(0)

    avg_train_loss = total_loss / len(train_loader)
    train_acc = correct_train / total_train

    model_mlp.eval()
    total_val_loss = 0
    correct_val = 0
    total_val = 0

    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            preds = model_mlp(x_batch)
            loss = loss_fn(preds, y_batch)
            total_val_loss += loss.item()

            predicted = torch.argmax(preds, dim=1)
            correct_val += (predicted == y_batch).sum().item()
            total_val += y_batch.size(0)

    avg_val_loss = total_val_loss / len(val_loader)
    val_acc = correct_val / total_val

    if val_acc >= 0.95:
        reached_acc_threshold = True

    status = ""
    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        best_val_acc = val_acc
        patience_counter = 0
        torch.save(model_mlp.state_dict(), "best_model.pth")
        status = " Saved best"
    else:
        patience_counter += 1
        if patience_counter >= patience:
            if reached_acc_threshold:
                status = " Early Stop (>99%)"
                print(f"\n{status} at Epoch {epoch}\n")
                break
            else:
                status = "  Keep Training (<99%)"

    # Print formatted row
    print(f"{epoch:<6} {avg_train_loss:<12.4f} {train_acc:<12.4f} {avg_val_loss:<12.4f} {val_acc:<12.4f} {status}")

print("\n Training complete.\n")



🚀 Starting training...

Epoch  Train Loss   Train Acc    Val Loss     Val Acc      Status
1      0.0640       0.9869       0.1729       0.9231        Saved best
2      0.0629       0.9869       0.2046       0.8974       
3      0.0636       0.9869       0.1875       0.8974       
4      0.0727       0.9869       0.2091       0.8974       
5      0.0613       0.9869       0.2322       0.8974       
6      0.0766       0.9739       0.1849       0.8974         Keep Training (<99%)
7      0.0602       0.9869       0.1681       0.9231        Saved best
8      0.0625       0.9869       0.2693       0.8718       
9      0.0687       0.9869       0.2686       0.9487       
10     0.0578       0.9935       0.1684       0.9231       
11     0.0738       0.9804       0.2548       0.9231       
12     0.0788       0.9739       0.1610       0.9231        Saved best
13     0.0495       0.9869       0.2069       0.9231       
14     0.0597       0.9804       0.1778       0.8974       
15     0.0560 

In [30]:
def predict(text):
    model_mlp.eval()
    with torch.no_grad():
        emb = get_xlmr_embedding(text).unsqueeze(0) 
        logits = model_mlp(emb)
        pred_idx = torch.argmax(logits, dim=1).item()
        return idx_to_label[pred_idx]

def predict_debug(text):
    model_mlp.eval()
    with torch.no_grad():
        emb = get_xlmr_embedding(text).unsqueeze(0)  
        logits = model_mlp(emb)
        probs = torch.softmax(logits, dim=1).squeeze()

        print("Prediction probabilities:")
        for i, p in enumerate(probs):
            print(f"  {idx_to_label[i]:<10}: {p:.4f}")

        pred_idx = torch.argmax(probs).item()
        return idx_to_label[pred_idx]


In [34]:
def classify_by_averaging(text, debug=False):
    chunks = split_text_to_chunks(text)

    if debug:
        print(f"📄 Total chunks: {len(chunks)}")

    chunk_embeddings = [get_xlmr_embedding(chunk) for chunk in chunks]
    avg_embedding = torch.stack(chunk_embeddings).mean(dim=0).unsqueeze(0)

    model_mlp.eval()
    with torch.no_grad():
        logits = model_mlp(avg_embedding)
        probs = torch.softmax(logits, dim=1).squeeze()
        pred_idx = torch.argmax(probs).item()

    if debug:
        print("\nFinal averaged probabilities:")
        for j, p in enumerate(probs):
            print(f"  {idx_to_label[j]:<10}: {p:.4f}")

    return probs, idx_to_label[pred_idx]


In [ ]:
import pandas as pd

test_df = pd.read_csv("../make-dataset/cleaned_test_website.csv", encoding='utf-8')


true_labels = ["게임", "보험", "기타"]  # Tossinsu, Poki

print(f" Loaded {len(test_df)} samples from test-website.csv")


for idx in range(len(test_df)):
    sample_test_text = test_df.loc[idx, "cleaned_website_text"]
    true_label = true_labels[idx]
    
    print(f"\nTesting Website {idx}: {test_df.loc[idx, 'website_url']}")
    print(f" True Label (expected): {true_label}")
    
    print("\nPrediction using averaged chunk embeddings:")
    probs, predicted_label = classify_by_averaging(sample_test_text, debug=True)

    # Apply threshold
    max_prob = probs.max().item()
    if max_prob < 0.6:
        predicted_label = "기타"  # Force to Unknown/Other if confidence too low

    print(f"\nPrediction Done: Predicted label = {predicted_label}")
    
    if predicted_label == true_label:
        print(f"Correct prediction!")
    else:
        print(f"Incorrect prediction! (Expected {true_label}, got {predicted_label})")
    
    print("-" * 80)

 Loaded 3 samples from test-website.csv

Testing Website 0: https://www.poki.com/
 True Label (expected): 게임

Prediction using averaged chunk embeddings:
📄 Total chunks: 6

Final averaged probabilities:
  게임        : 1.0000
  금융        : 0.0000
  보험        : 0.0000
  주식        : 0.0000
  코인        : 0.0000

Prediction Done: Predicted label = 게임
Correct prediction!
--------------------------------------------------------------------------------

Testing Website 1: https://dbinsure.co.kr/tome?JEHUSA_CD=C5450&utm_medium=cpc&utm_source=google&utm_campaign=pc&utm_content=tome&utm_term=%EB%B3%B4%ED%97%98&gad_source=1&gad_campaignid=21468186851&gclid=Cj0KCQjw2ZfABhDBARIsAHFTxGystt8yzAPTBc-D-g_kgKumE9mnwEmdu3OyeLcqwq7JwitBoSBO8tgaAh-zEALw_wcB
 True Label (expected): 보험

Prediction using averaged chunk embeddings:
📄 Total chunks: 8

Final averaged probabilities:
  게임        : 0.0000
  금융        : 0.0000
  보험        : 1.0000
  주식        : 0.0000
  코인        : 0.0000

Prediction Done: Predicted l